In [1]:
import pandas as pd
import re
import numpy as np
import os
import pickle
os.chdir('Z:/docker')
os.getcwd()

'Z:\\docker'

In [2]:
#  "DLBC" and "LAML" doesn't have drug response file in GDC.
study_list = ["ACC",    "BLCA",    "BRCA",    "CESC",    "CHOL",    "COAD",
    #"DLBC",
    "ESCA",    "GBM",    "HNSC",    "KICH",    "KIRC",    "KIRP",
    #"LAML",
    "LGG",    "LIHC",    "LUAD",    "LUSC",    "MESO",    "OV",    "PAAD",
    "PCPG",    "PRAD",    "READ",    "SARC",    "SKCM",    "STAD",    "TGCT",
    "THCA",    "THYM",    "UCEC",   "UCS",  "UVM"]
# study_list = ["ACC"]

# merge all file in one (all samples)

In [4]:
# Define a list of values that should be treated as missing (NaN)
NA_VALUES = [    "[Not Available]",    "[Not Applicable]",    "[Discrepancy]",
    "[Unknown]",    "[Undefined]",    "",    " ",    "  ",    "NA",    "NaN",
    "nan",    "NULL",    "null"]
def standardize_na(df):
    """
    Convert TCGA missing value strings into np.nan.
    """
    return df.replace(NA_VALUES, np.nan)

# normalize the response labels to 0 and 1, where 1 is response and 0 is non-response 
response_map = { 
    "COMPLETE RESPONSE": 1,
    "PARTIAL RESPONSE": 1,
    "STABLE DISEASE": 0,
    "PROGRESSIVE DISEASE": 0,
    "CLINICAL PROGRESSIVE DISEASE": 0,
    }

# normalize the drug name using the drug variant table from Ding2016 paper supplementary table
drug_varient = pd.read_csv("./data/TCGA/GDC_TCGA_drug_varient.csv") # from Ding2016 paper supplementary table
drug_varient = drug_varient.rename(    columns={"Recorded Name from TCGA": "pharmaceutical_therapy_drug_name"})
drug_varient = drug_varient.rename(    columns={"Standard drug name": "drug_name"})

# convert drug name to PubChem CID using PubChemPy
import pandas
from pubchempy import get_compounds
def drug_to_cid(name):
    for namespace in ["name", "synonym"]:
        try:
            compounds = get_compounds(name, namespace)
            if compounds:
                return compounds[0].cid
        except Exception:
            pass
    return None

# store all the TCGA study results in a list and concatenate them at the end
all_results = []

for study in study_list:
    print(f"\nProcessing {study}")
    drug = pd.read_csv(
        f"./data/TCGA/GDC_TCGA_response_rawdata/{study}/nationwidechildrens.org_clinical_drug_{study.lower()}.txt",
        sep="\t",        dtype=str    )
    sample = pd.read_csv(
        f"./data/TCGA/GDC_TCGA_response_rawdata/{study}/nationwidechildrens.org_biospecimen_sample_{study.lower()}.txt",
        sep="\t",        dtype=str    )
    patient = pd.read_csv(
        f"./data/TCGA/GDC_TCGA_response_rawdata/{study}/nationwidechildrens.org_clinical_patient_{study.lower()}.txt",
        sep="\t",        dtype=str    )
    radiation = pd.read_csv(
        f"./data/TCGA/GDC_TCGA_response_rawdata/{study}/nationwidechildrens.org_clinical_radiation_{study.lower()}.txt",
        sep="\t",        dtype=str    )
    drug = standardize_na(drug)[2:]
    sample = standardize_na(sample)[1:]
    patient = standardize_na(patient)[2:]
    radiation = standardize_na(radiation)[2:]
    # print(drug.columns)
    # print(sample.columns)
    # print(patient.columns)
    # print(radiation.columns)

    # Add missing columns to patient dataframe if they don't exist # some TCGA studies don't have these columns, so we add them with NaN values to ensure consistent data structure across all studies
    for col in ["history_neoadjuvant_steroid_tx", "history_neoadjuvant_medication", "treatment_outcome_first_course"]:
        if col not in patient.columns:
            patient[col] = np.nan

    # Rename the column in radiation dataframe if it exists
    if "days_to_radiation_therapy_start" in radiation.columns:
        radiation = radiation.rename(    columns={"days_to_radiation_therapy_start": "radiation_therapy_started_days_to"})
    # 如果一個病人有多次radiation day，只取最小的 # 因為要和drug treatment day比較先後
    radiation = radiation.sort_values('radiation_therapy_started_days_to').drop_duplicates(subset='bcr_patient_barcode', keep='first').reset_index(drop=True)

    sample = sample[sample["bcr_sample_barcode"].str.split("-").str[3].str[:2].isin(["1", "01"])]#只保留 primary tumor
    sample["bcr_patient_barcode"]=sample["bcr_sample_barcode"].str[0:12]
    drug["treatment_best_response"] = (drug["treatment_best_response"].astype(str).str.strip().str.upper()) # 統一RECIST大小寫
    drug = drug[drug["treatment_best_response"].isin(response_map.keys())]# 篩掉treatment_best_response不屬於任何一個key的samples
    drug["response_label"] = (drug["treatment_best_response"].map(response_map)) # convert RECIST to binary labels
    # print("drug shape before merge:",drug.shape)
    # 將drug_varient藥名修正表的drug_name以pharmaceutical_therapy_drug_name merge到drug
    drug = drug.merge(drug_varient[["pharmaceutical_therapy_drug_name", "drug_name"]]
                  , on="pharmaceutical_therapy_drug_name", how="left") 
    # print("drug shape after merge:",drug.shape)
    unique_drugs = drug["drug_name"].dropna().unique()#　列出所有藥物名稱，並去掉NA值
    # map drug name to PubChem CID
    drug2cid = {} 
    for d in unique_drugs:
        drug2cid[d] = drug_to_cid(d)
    drug["PubChemID"] = drug["drug_name"].map(drug2cid)
    
    merge_df = drug.merge(sample[["bcr_patient_barcode", "days_to_collection"]], on="bcr_patient_barcode", how="left" )#將days_to_collection merge到drug
    # print("merge_df shape after merge sample:",merge_df.shape)
    merge_df = merge_df.merge(patient[["bcr_patient_barcode", 
                                        "history_neoadjuvant_treatment",
                                        "radiation_treatment_adjuvant",
                                        "treatment_outcome_first_course",
                                        "history_neoadjuvant_steroid_tx",
                                        "history_neoadjuvant_medication",]]
                                        , on="bcr_patient_barcode", how="left" )
    # print("merge_df shape after merge patient:",merge_df.shape)
    merge_df = merge_df.merge(radiation[["bcr_patient_barcode", 
                                        "radiation_therapy_started_days_to"]]
                                        , on="bcr_patient_barcode", how="left" )
    # print("merge_df shape after merge radiation:",merge_df.shape)
    merge_df["TCGA_study"]=study.upper()
    # Reorder columns 
    merge_df = merge_df[
        ["TCGA_study", "bcr_patient_barcode", "PubChemID", "response_label","pharmaceutical_therapy_drug_name", "pharmaceutical_therapy_type",
         "treatment_best_response", "days_to_collection", "pharmaceutical_tx_started_days_to", "pharmaceutical_tx_ended_days_to", 
         "radiation_therapy_started_days_to", "radiation_treatment_adjuvant", "history_neoadjuvant_treatment", "history_neoadjuvant_steroid_tx",
         "history_neoadjuvant_medication","treatment_outcome_first_course"
        ]               ]
    all_results.append(merge_df)
combined_df = pd.concat(all_results,ignore_index=True)
combined_df.to_csv("./data/TCGA/GDC_TCGA_drug_response_all_samples.csv",index=False)
    


Processing ACC


C:\Users\yukai\AppData\Local\Temp\ipykernel_23892\3120224896.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(NA_VALUES, np.nan)



Processing BLCA


C:\Users\yukai\AppData\Local\Temp\ipykernel_23892\3120224896.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(NA_VALUES, np.nan)



Processing BRCA


...

C:\Users\yukai\AppData\Local\Temp\ipykernel_23892\3120224896.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(NA_VALUES, np.nan)



Processing UCS


C:\Users\yukai\AppData\Local\Temp\ipykernel_23892\3120224896.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(NA_VALUES, np.nan)



Processing UVM


C:\Users\yukai\AppData\Local\Temp\ipykernel_23892\3120224896.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(NA_VALUES, np.nan)


目標是建立：
Tumor expression (治療前) -> Single-drug response (RECIST)

因此我建議的完整流程如下：

Step 1. Primary Tumor
保留：sample_type == "Primary Tumor"
排除：
Blood Derived Normal
Metastatic
Recurrent Tumor
理由：確保 expression 來自原發腫瘤。

Step 2. RECIST 轉換
Response 僅接受：
Complete Response → 1
Partial Response → 1
Stable Disease → 0
Progressive Disease → 0
Clinical Progressive Disease → 0
新增：response_groundtruth

Step 3. 排除多藥重疊治療
例如：
TMZ
10~100
Avastin
50~200
重疊：50~100
直接刪除病人。
理由：RECIST 無法歸因於單藥。

Step 4. 排除 Neoadjuvant Treatment
保留：history_neoadjuvant_treatment == NO
排除：YES
理由：不希望 expression 已經受到治療影響。

Step 5. Expression 必須早於 Drug
保留：days_to_collection < days_to_drug_therapy_start
例如：Collection = 5, Drug Start = 30, 保留。
例如：Collection = 100, Drug Start = 30, 刪除。
理由：確保 expression 是 baseline。

Step 6. Radiation 必須晚於 Drug  
if 如果沒有Radiation treatment:if radiation_treatment_adjuvant == NO
直接保留sample
elif 如果有Radiation treatment: (包括radiation_treatment_adjuvant==YES和其他)
保留：pharmaceutical_tx_ended_days_to < first_radiation_start
理由：radiation 會影響RECIST

Step 7. 每病人只保留第一個符合條件的藥
排序：days_to_drug_therapy_start
保留最早開始的藥：groupby(patient).head(1)
理由：避免後續藥物受到前面治療影響。

Step 8. 同病人同藥衝突 
只取最後一個時段的藥效結果，但是容易遇到其他藥物在其之前使用，所以這個標記不是fatal
例如：
patient	drug	RECIST	end
A	TMZ	SD	100
A	TMZ	PD	250
保留：end=250, PD
規則：同病人同藥, 保留治療結束時間最晚pharmaceutical_tx_ended_days_to，由於step 1~5已經篩除缺失判斷依據的samples，所以不用擔心此時最晚的sample會有依據缺失


目標是建立：
Tumor expression (治療前) -> Single-drug response (RECIST)

因此我建議的完整流程如下：

Step 1. Primary Tumor
保留：sample_type == "Primary Tumor"
排除：
Blood Derived Normal
Metastatic
Recurrent Tumor
理由：確保 expression 來自原發腫瘤。

Step 2. 排除 Neoadjuvant Treatment
保留：history_neoadjuvant_treatment == NO
排除：YES
理由：不希望 expression 已經受到治療影響。

Step 3. Expression 必須早於 Drug
保留：days_to_collection < days_to_drug_therapy_start
例如：Collection = 5, Drug Start = 30, 保留。
例如：Collection = 100, Drug Start = 30, 刪除。
理由：確保 expression 是 baseline。

Step 4. Radiation 必須晚於 Drug  
if 如果沒有Radiation treatment:if radiation_treatment_adjuvant == NO
直接保留sample
elif 如果有Radiation treatment: (包括radiation_treatment_adjuvant==YES和其他)
保留：pharmaceutical_tx_ended_days_to < first_radiation_start
理由：radiation 會影響RECIST


Step 5. RECIST 轉換
Response 僅接受：
Complete Response → 1
Partial Response → 1
Stable Disease → 0
Progressive Disease → 0
Clinical Progressive Disease → 0
新增：response_groundtruth

Step 6. 同病人同藥衝突
例如：
patient	drug	RECIST	end
A	TMZ	SD	100
A	TMZ	PD	250
保留：end=250, PD
規則：同病人同藥, 保留治療結束時間最晚pharmaceutical_tx_ended_days_to，由於step 1~5已經篩除缺失判斷依據的samples，所以不用擔心此時最晚的sample會有依據缺失

Step 7. 排除多藥重疊治療
例如：
TMZ
10~100
Avastin
50~200
重疊：50~100
直接刪除病人。
理由：RECIST 無法歸因於單藥。

Step 8. 每病人只保留第一個符合條件的藥
排序：days_to_drug_therapy_start
保留最早開始的藥：groupby(patient).head(1)
理由：避免後續藥物受到前面治療影響。


# annotation problem for all samples

- no drug day
- no drug name
- no RECIST Criteria
- Drug Overlap
- Neoadjuvant Treatment
- Collection after Drug
- Radiation Treatment
- Not First Drug      when multiple drug but not the same time 
- Same Patient Same Drug只取最後一個時段的藥效結果，但是容易遇到其他藥物在其之前使用，所以這個標記不是fatal

In [5]:
import pandas as pd
import re
import numpy as np
import os
import pickle
os.chdir('Z:/docker')
os.getcwd()
df = pd.read_csv("./data/TCGA/GDC_TCGA_drug_response_all_samples_raw-test.csv")
df

,TCGA_study,bcr_patient_barcode,PubChemID,response_label,pharmaceutical_therapy_drug_name,pharmaceutical_therapy_type,treatment_best_response,days_to_collection,pharmaceutical_tx_started_days_to,pharmaceutical_tx_ended_days_to,radiation_therapy_started_days_to,radiation_treatment_adjuvant,history_neoadjuvant_treatment,history_neoadjuvant_steroid_tx,history_neoadjuvant_medication,treatment_outcome_first_course
0,ACC,TCGA-OR-A5JM,5329102.0,0,sunitinib,Targeted Molecular therapy,CLINICAL PROGRESSIVE DISEASE,657.0,378,439,134.0,YES,Yes,NaN,NaN,Progressive Disease
1,ACC,TCGA-OR-A5JM,47576.0,0,ketoconazole,Targeted Molecular therapy,CLINICAL PROGRESSIVE DISEASE,657.0,378,439,134.0,YES,Yes,NaN,NaN,Progressive Disease
2,ACC,TCGA-OU-A5PI,36462.0,0,Etoposide,Chemotherapy,STABLE DISEASE,577.0,69,239,476.0,NO,No,NaN,NaN,Stable Disease
3,ACC,TCGA-OU-A5PI,31703.0,0,doxorubicin,Chemotherapy,STABLE DISEASE,577.0,69,239,476.0,NO,No,NaN,NaN,Stable Disease
4,ACC,TCGA-OU-A5PI,5702198.0,0,cisplatin,Chemotherapy,STABLE DISEASE,577.0,55,239,476.0,NO,No,NaN,NaN,Stable Disease
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3882,UCS,TCGA-NG-A4VU,36314.0,0,Taxol,Chemotherapy,CLINICAL PROGRESSIVE DISEASE,500.0,76,258,NaN,NO,No,NaN,NaN,Progressive Disease
3883,UCS,TCGA-NG-A4VU,3690.0,0,Ifosfamide,Chemotherapy,CLINICAL PROGRESSIVE DISEASE,500.0,76,258,NaN,NO,No,NaN,NaN,Progressive Disease
3884,UCS,TCGA-NG-A4VW,5702198.0,1,Cisplatin,Chemotherapy,COMPLETE RESPONSE,2264.0,57,197,58.0,YES,No,NaN,NaN,Complete Remission/Response
3885,UCS,TCGA-NG-A4VW,31703.0,1,Adriamycin,Chemotherapy,COMPLETE RESPONSE,2264.0,57,197,58.0,YES,No,NaN,NaN,Complete Remission/Response


In [6]:
def check_overlap(group):# 檢查同一病人不同藥物的治療期間是否有重疊    
    n = len(group) # 計算同一病人有幾筆藥物紀錄
    problems = pd.Series([None] * n, index=group.index) # 建立一個新的 Series，長度和病人紀錄一樣，初始值都是 None，用來存放每筆紀錄的問題標記。
    for i in range(n): # 遍歷每一筆藥物紀錄
        for j in range(i+1, n): # sample i 和 j 比較
            if group.iloc[i]['PubChemID'] != group.iloc[j]['PubChemID']:
                start_i, end_i = group.iloc[i]['pharmaceutical_tx_started_days_to'], group.iloc[i]['pharmaceutical_tx_ended_days_to']
                start_j, end_j = group.iloc[j]['pharmaceutical_tx_started_days_to'], group.iloc[j]['pharmaceutical_tx_ended_days_to']
                # 確保不是 NaN 才比較
                if pd.notna(start_i) and pd.notna(end_i) and pd.notna(start_j) and pd.notna(end_j):
                    if (start_i <= end_j) and (start_j <= end_i):
                        problems.iloc[i] = "drug overlap"
                        problems.iloc[j] = "drug overlap"
    return problems

#no drug day
mask = df['pharmaceutical_tx_started_days_to'].isna() & df['pharmaceutical_tx_ended_days_to'].isna()
df.loc[mask, 'no drug day'] = "no drug day"

#no drug CID
mask = df['PubChemID'].isna()
df.loc[mask, 'no drug CID'] = "no drug CID"

#no RECIST Criteria
mask = df['treatment_best_response'].isna()
df.loc[mask, 'no RECIST Criteria'] = "no RECIST Criteria"

#drug overlap
df['drug overlap'] = df.groupby('bcr_patient_barcode', group_keys=False).apply(check_overlap)

#Neoadjuvant is yes
mask = df['history_neoadjuvant_treatment'].str.upper() == "YES"
df.loc[mask, 'Neoadjuvant'] = "Neoadjuvant YES"

#Collect day after Drug
df['days_to_collection'] = pd.to_numeric(df['days_to_collection'], errors='coerce')# 確保欄位是數值型態
df['pharmaceutical_tx_started_days_to'] = pd.to_numeric(df['pharmaceutical_tx_started_days_to'], errors='coerce')
mask = df['days_to_collection'] > df['pharmaceutical_tx_started_days_to']
# 標記 problem 欄位
df.loc[mask, 'Collect after Drug'] = "Collect after Drug"

#Radiation Treatment
df['radiation_therapy_started_days_to'] = pd.to_numeric(df['radiation_therapy_started_days_to'], errors='coerce')
df['pharmaceutical_tx_ended_days_to'] = pd.to_numeric(df['pharmaceutical_tx_ended_days_to'], errors='coerce')
mask = df['radiation_therapy_started_days_to'].isna() 
df.loc[mask, 'radiation'] = "no radiation day"
mask = df['radiation_therapy_started_days_to'] < df['pharmaceutical_tx_ended_days_to']
df.loc[mask, 'radiation'] = "radiation before drug"

#Not First Drug
df['pharmaceutical_tx_started_days_to'] = pd.to_numeric(df['pharmaceutical_tx_started_days_to'], errors='coerce')
df['min_drug'] = df.groupby('bcr_patient_barcode')['pharmaceutical_tx_started_days_to'].transform('min')# 找出同一病人不同藥物的最小 pharmaceutical_tx_started_days_to
df.loc[df['pharmaceutical_tx_started_days_to'] > df['min_drug'], 'Not First Drug'] = "Not First Drug"# 標記：不是最小值的就標記 "not first drug"
df.drop(columns=['min_drug'], inplace=True)

#Same Patient Same Drug # 取最後的那筆結果作為groundtruth，但是容易遇到其他藥物在其之前使用，所以這個標記不是fatal
df['pharmaceutical_tx_ended_days_to'] = pd.to_numeric(df['pharmaceutical_tx_ended_days_to'], errors='coerce')# 確保欄位是數值型態
df['max_end'] = df.groupby(['bcr_patient_barcode','PubChemID'])['pharmaceutical_tx_ended_days_to'].transform('max')# 找出同一病人同一藥物的最大結束日
df.loc[df['pharmaceutical_tx_ended_days_to'] < df['max_end'], 'Same Patient Same Drug'] = "Same Patient Same Drug"# 標記問題：不是最大值的就標記問題
df.drop(columns=['max_end'], inplace=True)# 如果不需要 max_end 欄位，可以刪掉

df


C:\Users\yukai\AppData\Local\Temp\ipykernel_23892\3974612073.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['drug overlap'] = df.groupby('bcr_patient_barcode', group_keys=False).apply(check_overlap)


,TCGA_study,bcr_patient_barcode,PubChemID,response_label,pharmaceutical_therapy_drug_name,pharmaceutical_therapy_type,treatment_best_response,days_to_collection,pharmaceutical_tx_started_days_to,pharmaceutical_tx_ended_days_to,...,treatment_outcome_first_course,no drug day,no drug CID,no RECIST Criteria,drug overlap,Same Patient Same Drug,Neoadjuvant,Collect after Drug,radiation,Not First Drug
0,ACC,TCGA-OR-A5JM,5329102.0,0,sunitinib,Targeted Molecular therapy,CLINICAL PROGRESSIVE DISEASE,657.0,378.0,439.0,...,Progressive Disease,NaN,NaN,NaN,drug overlap,NaN,Neoadjuvant YES,Collect after Drug,radiation before drug,NaN
1,ACC,TCGA-OR-A5JM,47576.0,0,ketoconazole,Targeted Molecular therapy,CLINICAL PROGRESSIVE DISEASE,657.0,378.0,439.0,...,Progressive Disease,NaN,NaN,NaN,drug overlap,NaN,Neoadjuvant YES,Collect after Drug,radiation before drug,NaN
2,ACC,TCGA-OU-A5PI,36462.0,0,Etoposide,Chemotherapy,STABLE DISEASE,577.0,69.0,239.0,...,Stable Disease,NaN,NaN,NaN,None,Same Patient Same Drug,NaN,Collect after Drug,NaN,Not First Drug
3,ACC,TCGA-OU-A5PI,31703.0,0,doxorubicin,Chemotherapy,STABLE DISEASE,577.0,69.0,239.0,...,Stable Disease,NaN,NaN,NaN,None,Same Patient Same Drug,NaN,Collect after Drug,NaN,Not First Drug
4,ACC,TCGA-OU-A5PI,5702198.0,0,cisplatin,Chemotherapy,STABLE DISEASE,577.0,55.0,239.0,...,Stable Disease,NaN,NaN,NaN,None,NaN,NaN,Collect after Drug,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3882,UCS,TCGA-NG-A4VU,36314.0,0,Taxol,Chemotherapy,CLINICAL PROGRESSIVE DISEASE,500.0,76.0,258.0,...,Progressive Disease,NaN,NaN,NaN,None,NaN,NaN,Collect after Drug,no radiation day,NaN
3883,UCS,TCGA-NG-A4VU,3690.0,0,Ifosfamide,Chemotherapy,CLINICAL PROGRESSIVE DISEASE,500.0,76.0,258.0,...,Progressive Disease,NaN,NaN,NaN,None,NaN,NaN,Collect after Drug,no radiation day,NaN
3884,UCS,TCGA-NG-A4VW,5702198.0,1,Cisplatin,Chemotherapy,COMPLETE RESPONSE,2264.0,57.0,197.0,...,Complete Remission/Response,NaN,NaN,NaN,None,NaN,NaN,Collect after Drug,radiation before drug,NaN
3885,UCS,TCGA-NG-A4VW,31703.0,1,Adriamycin,Chemotherapy,COMPLETE RESPONSE,2264.0,57.0,197.0,...,Complete Remission/Response,NaN,NaN,NaN,None,NaN,NaN,Collect after Drug,radiation before drug,NaN


In [22]:
df.to_csv("./data/TCGA/GDC_TCGA_drug_response_all_samples_with_problems.csv", index=False)